# WatSPEED Prep Module 1: Python Data & API Essentials for SAS/Stata Veterans

Welcome! As a sociologist experienced in **SAS** (`DATA` steps, `PROC` procedures) and **Stata** (`.do` files, `recode`), this notebook bridges your analytical intuition to **Python, Pandas, Pydantic, and Async LLM APIs**.

---

### 📖 Concept Deep-Dive & Terminology Breakdown

> **What is Pydantic?**
> * **Definition**: Pydantic is Python's standard data validation library. It enforces data types (e.g. integer, string, boolean) using Python type hints.
> * **SAS/Stata Analogy**: Think of Pydantic like combining `PROC FORMAT` validation rules with SAS `DATA` step `ERROR` log checks. It acts as a 'bouncer' ensuring data passed to or from an LLM model adheres to strict schemas.
> * **Why Agents Need It**: LLMs output freeform text. Pydantic guarantees that numbers are parsed as actual integers, missing values are flagged as `None`, and invalid schemas raise explicit errors.

> **What is Asyncio?**
> * **Definition**: Asynchronous I/O framework that allows Python to execute multiple non-blocking tasks concurrently.
> * **SAS Analogy**: Submitting 50 SAS batch jobs in parallel rather than running them sequentially one after another.

---

### SAS/Stata vs Python Mental Model:
* **SAS DATA Step / Stata recode** -> **Pandas DataFrame manipulation & Pydantic Data Validation Models**
* **SAS PROC FREQ / PROC SURVEYREG** -> **Pandas value_counts() & scikit-learn weighted models**
* **Dataset Selection**: You can toggle between **GSS Social Survey** (`data/gss_survey_data.csv`) and **AI Trust Insights** (`data/ai_trust_insights.csv`) below!



In [1]:
import pandas as pd
import numpy as np
import json
import asyncio
from pydantic import BaseModel, Field
from typing import List, Optional

# Load survey datasets
gss_df = pd.read_csv('../data/gss_survey_data.csv')
ai_trust_df = pd.read_csv('../data/ai_trust_insights.csv')

print(f"GSS Dataset Loaded: {gss_df.shape[0]} respondents, {gss_df.shape[1]} columns")
print(f"AI Trust Dataset Loaded: {ai_trust_df.shape[0]} respondents, {ai_trust_df.shape[1]} columns")

print("\n--- GSS Sample (SAS PROC PRINT data=gss(obs=3); run;) ---")
print(gss_df[['Respondent_ID', 'Education_Degree', 'Political_Views', 'High_Institutional_Trust']].head(3))

print("\n--- AI Trust Sample ---")
print(ai_trust_df[['Respondent_ID', 'Education_Level', 'Employment_Sector', 'High_AI_Trust']].head(3))



GSS Dataset Loaded: 1200 respondents, 11 columns
AI Trust Dataset Loaded: 1200 respondents, 16 columns

--- GSS Sample (SAS PROC PRINT data=gss(obs=3); run;) ---
  Respondent_ID Education_Degree Political_Views  High_Institutional_Trust
0      GSS_2000      High School         Liberal                         1
1      GSS_2001         Graduate        Moderate                         0
2      GSS_2002      High School        Moderate                         0

--- AI Trust Sample ---
  Respondent_ID Education_Level Employment_Sector  High_AI_Trust
0     RESP_1000        Master's        Healthcare              1
1     RESP_1001     High School  Tech/Engineering              1
2     RESP_1002     High School         Education              0


### 1.1 SAS DATA Step Equivalent: Cleaning & Recoding Missing Values (-9)

In SAS you might write:
```sas
data clean_ai;
    set ai_trust_insights;
    if Perceived_AI_Risk = -9 then Perceived_AI_Risk = .;
    High_Risk = (Perceived_AI_Risk >= 4);
run;
```

In Python with Pandas:



In [2]:
# Replace missing codes (-9) with NaN or mean imputation
clean_ai = ai_trust_df.copy()
clean_ai['Perceived_AI_Risk_Clean'] = clean_ai['Perceived_AI_Risk'].replace(-9, np.nan)
clean_ai['Perceived_AI_Benefit_Clean'] = clean_ai['Perceived_AI_Benefit'].replace(-9, np.nan)

# Create binary indicator
clean_ai['High_Risk_Perception'] = (clean_ai['Perceived_AI_Risk_Clean'] >= 4).astype(int)

print("Cross-tabulation (SAS PROC FREQ / Stata tabulate):")
print(pd.crosstab(clean_ai['Education_Level'], clean_ai['High_Risk_Perception'], margins=True))



Cross-tabulation (SAS PROC FREQ / Stata tabulate):
High_Risk_Perception    0    1   All
Education_Level                     
Bachelor's            374  230   604
High School           152   95   247
Master's              155   87   242
PhD                    69   38   107
All                   750  450  1200


### 1.2 Pydantic Data Models: Schema Enforcement for Agent Inputs


In [3]:
class SurveyRespondent(BaseModel):
    respondent_id: str
    education_level: str
    employment_sector: str
    tech_familiarity: str
    perceived_risk_score: Optional[int] = Field(default=None, description="1-5 Likert score")
    high_ai_trust: bool

sample_row = clean_ai.iloc[0]
respondent = SurveyRespondent(
    respondent_id=sample_row['Respondent_ID'],
    education_level=sample_row['Education_Level'],
    employment_sector=sample_row['Employment_Sector'],
    tech_familiarity=sample_row['Tech_Familiarity'],
    perceived_risk_score=int(sample_row['Perceived_AI_Risk_Clean']) if not pd.isna(sample_row['Perceived_AI_Risk_Clean']) else None,
    high_ai_trust=bool(sample_row['High_AI_Trust'])
)

print("Pydantic Verified Model JSON Output:")
print(respondent.model_dump_json(indent=2))



Pydantic Verified Model JSON Output:
{
  "respondent_id": "RESP_1000",
  "education_level": "Master's",
  "employment_sector": "Healthcare",
  "tech_familiarity": "Novice",
  "perceived_risk_score": 1,
  "high_ai_trust": true
}


### 1.3 Asynchronous API Calls (Python asyncio for Parallel LLM Queries)


In [4]:
async def mock_llm_summarizer(info_str):
    await asyncio.sleep(0.05)
    return f"Sociological Profile [{info_str}]: Driven by institutional trust & tech background."

async def run_batch(df_slice):
    tasks = [mock_llm_summarizer(f"{r['Education_Level']} in {r['Employment_Sector']}") for _, r in df_slice.iterrows()]
    return await asyncio.gather(*tasks)

results = await run_batch(clean_ai.head(5))
for res in results:
    print(" ->", res)



 -> Sociological Profile [Master's in Healthcare]: Driven by institutional trust & tech background.
 -> Sociological Profile [High School in Tech/Engineering]: Driven by institutional trust & tech background.
 -> Sociological Profile [High School in Education]: Driven by institutional trust & tech background.
 -> Sociological Profile [Bachelor's in Healthcare]: Driven by institutional trust & tech background.
 -> Sociological Profile [Bachelor's in Finance]: Driven by institutional trust & tech background.
